<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Juego de las 20 Preguntas: Árboles de Decisión 🌳
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/02%20-%20Arboles%20Bosques%20y%20Redes%20Neuronales/Para%20Dummies/00_Arboles_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del cuaderno 00 (Árboles de Decisión). ¿Has jugado a las **20 preguntas** o a *«¿Quién es quién?»*? Vas adivinando haciendo preguntas de sí o no: *«¿Tiene lentes?», «¿Es hombre?»*. Un **árbol de decisión** hace exactamente eso con los datos: aprende **qué preguntas hacer y en qué orden** para llegar a una respuesta.

Al terminar podrás explicar:
1. Cómo un **árbol** toma decisiones con **preguntas de sí o no**.
2. Cómo elige **qué pregunta** hacer primero (la que mejor «ordena» los casos).
3. Por qué un árbol con **demasiadas preguntas memoriza** en vez de aprender.
4. Por qué un árbol **cambia mucho** si cambian pocos datos.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../00_Arboles_de_Decision.ipynb). Otras miradas sencillas: [CART (Dummies) en Data Mining](../../../Data%20Mining/04%20-%20Arboles%20de%20Decision%20y%20Bosques%20Aleatorios/Para%20Dummies/00_CART_Dummies.ipynb) y [Introducción a Decision Trees (Dummies) en Data Science Programming](../../../Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/00_Introduccion_Decision_Trees_Dummies.ipynb).

---
## 1. Un árbol es una cadena de preguntas ❓

Imagina que cada mañana quieres decidir **si llevar paraguas**. Un amigo meteorólogo te da una regla en forma de preguntas:

- *«¿Hay más de 60 % de nubes?»* → si **no**, no lleves paraguas.
- Si **sí**: *«¿La humedad pasa de 70 %?»* → si **sí**, lleva paraguas; si **no**, probablemente no llueve.

Eso es un **árbol de decisión**: cada **pregunta** es una *bifurcación* y cada **respuesta final** (llevar o no el paraguas) es una *hoja*. Lo bonito es que **se puede leer**: no hay fórmulas misteriosas, solo reglas de «si… entonces…».

Vamos a **inventar 200 días** (con un poco de azar, porque el clima real nunca es perfecto) y dejar que el computador **descubra solo** las preguntas.

In [ ]:
# Datos inventados: 200 dias con nubosidad y humedad; ¿llovio? (la regla oculta tiene un 10 % de azar)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
nubes = rng.uniform(0, 100, 200)                                      # % de nubosidad
humedad = rng.uniform(30, 100, 200)                                   # % de humedad
llovio = ((nubes > 60) & (humedad > 70)).astype(int)                  # la "regla oculta"
cambiar = rng.random(200) < 0.10                                      # 10 % de dias donde la regla falla (azar)
llovio = np.where(cambiar, 1 - llovio, llovio)

dias = pd.DataFrame({"nubes (%)": nubes.round(0), "humedad (%)": humedad.round(0), "llovió": llovio})
print(dias.head(5).to_string(index=False))

X = dias[["nubes (%)", "humedad (%)"]]
X_ent, X_exa, y_ent, y_exa = train_test_split(X, dias["llovió"], test_size=0.3, random_state=42)

arbol = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_ent, y_ent)
print("\nLas preguntas que descubrió el árbol:\n")
print(export_text(arbol, feature_names=list(X.columns)))
print("(class: 1 = llovió, class: 0 = no llovió)")

plt.figure(figsize=(11, 5))
plot_tree(arbol, feature_names=list(X.columns), class_names=["no llovió", "llovió"], filled=True, rounded=True, fontsize=9)
plt.title("El árbol que aprendió el computador", fontweight="bold")
plt.show()
print(f"\nAciertos en días que NO vio al entrenar: {arbol.score(X_exa, y_exa):.0%}")

---
## 2. ¿Cómo elige la mejor pregunta? 🧺

Piensa en una **bolsa con frijoles negros y blancos mezclados**. Quieres separarlos en dos bolsas **lo más «puros» posible** (una casi toda negra y otra casi toda blanca). Una buena pregunta es la que deja las bolsas **más ordenadas**; una mala pregunta las deja igual de revueltas.

El árbol hace esto **con todas las preguntas posibles** («¿nubes > 10 %?», «¿nubes > 11 %?», «¿humedad > 55 %?»…), mide **qué tan ordenado queda cada grupo** y **se queda con la mejor**. Luego repite el mismo juego **dentro de cada bolsa**. Esa es toda la idea.

Veamos cuánto «orden» gana cada pregunta posible sobre las nubes: la mejor es la **más baja** de la curva.

In [ ]:
# Para cada pregunta "nubes > t?", ¿que tan revueltos quedan los dos grupos? (menor = mas ordenado)
def revuelto(grupo):
    """0 = grupo puro (todos iguales); 0.5 = mitad y mitad (maxima mezcla)."""
    if len(grupo) == 0:
        return 0
    p = grupo.mean()
    return 2 * p * (1 - p) 

cortes = np.arange(5, 96, 1)
resultado = []
for corte in cortes:
    izq = y_ent[X_ent["nubes (%)"] <= corte]
    der = y_ent[X_ent["nubes (%)"] > corte]
    # promedio de la mezcla de los dos grupos, ponderado por cuantos dias tiene cada uno
    resultado.append((len(izq) * revuelto(izq) + len(der) * revuelto(der)) / len(y_ent))

mejor = cortes[int(np.argmin(resultado))]
plt.figure(figsize=(9, 4.5))
plt.plot(cortes, resultado, color="#f59e0b", lw=3)
plt.axvline(mejor, color="#16a34a", linestyle="--", lw=2, label=f"Mejor pregunta: ¿nubes > {mejor} %?")
plt.xlabel("Pregunta: ¿nubes (%) mayores que...?")
plt.ylabel("Qué tan revueltos quedan los grupos (menos = mejor)")
plt.title("El árbol prueba todas las preguntas y elige la que más ordena", fontweight="bold")
plt.legend()
plt.show()
print(f"La mejor pregunta sobre las nubes: ¿nubes > {mejor} %?  (la regla oculta que inventamos usaba 60 %)")

---
## 3. Demasiadas preguntas = memorizar 📖

¿Qué pasa si dejamos que el árbol haga **todas las preguntas que quiera**? Puede terminar con una pregunta **para cada día** del cuaderno: *«si las nubes están entre 63.2 y 63.4 % y la humedad entre 81 y 82 %, llovió»*. Es como el estudiante que **memoriza las respuestas del examen del año pasado**: saca 100 en esos ejercicios, pero **se confunde con preguntas nuevas**.

Con pocas preguntas, el árbol **no aprende lo suficiente**. Con demasiadas, **memoriza hasta el azar**. Lo sensato es un punto medio. Comparemos con árboles de distinto tamaño (profundidad = cuántas preguntas seguidas puede hacer).

In [ ]:
# Aciertos con los dias que VIO (entrenamiento) y con los que NO vio (examen), segun cuantas preguntas seguidas se permiten
filas = []
for profundidad in [1, 2, 3, 5, 8, None]:
    a = DecisionTreeClassifier(max_depth=profundidad, random_state=42).fit(X_ent, y_ent)
    filas.append({"preguntas seguidas": "las que quiera" if profundidad is None else profundidad,
                  "hojas": a.get_n_leaves(),
                  "aciertos que vio": a.score(X_ent, y_ent),
                  "aciertos en el examen": a.score(X_exa, y_exa)})
tabla = pd.DataFrame(filas)
print(tabla.round(2).to_string(index=False))

plt.figure(figsize=(8.5, 4.8))
x = range(len(tabla))
plt.plot(x, tabla["aciertos que vio"], "o-", color="#f59e0b", lw=3, label="Con lo que ya vio (memoria)")
plt.plot(x, tabla["aciertos en el examen"], "s-", color="#16a34a", lw=3, label="En el examen (días nuevos)")
plt.xticks(x, tabla["preguntas seguidas"])
plt.xlabel("Preguntas seguidas permitidas")
plt.ylabel("Proporción de aciertos")
plt.title("Memorizar sube los aciertos 'conocidos', no los del examen", fontweight="bold")
plt.legend()
plt.show()
sin_limite, mejor_fila = tabla.iloc[-1], tabla.loc[tabla["aciertos en el examen"].idxmax()]
print(f"\nEl árbol sin límite acierta {sin_limite['aciertos que vio']:.0%} de lo que vio, pero en el examen solo {sin_limite['aciertos en el examen']:.0%} "
      f"(el mejor llegó a {mejor_fila['aciertos en el examen']:.0%} con pocas preguntas): memorizó el azar.")

---
## 4. Un árbol cambia si cambian pocos datos 🎲

Hay otra característica curiosa: los árboles son **muy sensibles**. Si quitamos unos pocos días de los datos, **la primera pregunta puede cambiar por completo**. Es como un detective que, al perder una pista, arma **otra historia** distinta aunque llegue a conclusiones parecidas.

No es un defecto del programa: es que cada pregunta se elige mirando **solo los datos que llegaron a ese punto**. Y esta inestabilidad es justo lo que se aprovecha en el **siguiente cuaderno** (*bosques*): en vez de confiar en **un** árbol caprichoso, se consulta a **muchos**.

In [ ]:
# Entrenamos 8 arboles, cada uno con solo 35 dias elegidos al azar, y miramos su primera pregunta
rng = np.random.default_rng(3)
primeras = []
for i in range(8):
    idx = rng.choice(len(X_ent), size=35, replace=False)
    a = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_ent.iloc[idx], y_ent.iloc[idx])
    variable = X.columns[a.tree_.feature[0]]
    primeras.append(variable)
    print(f"Árbol {i + 1}: primera pregunta -> ¿{variable} > {a.tree_.threshold[0]:.1f}?   | aciertos en el examen: {a.score(X_exa, y_exa):.0%}")
print(f"\nLa primera pregunta fue sobre 'nubes' en {primeras.count('nubes (%)')} árboles y sobre 'humedad' en {primeras.count('humedad (%)')}, y el umbral cambia cada vez.")
print("Con datos distintos salen árboles distintos (y con aciertos distintos). ¡Un solo árbol es inestable!")

---
##### 🛠️ Práctica 1: Crea tu árbol sensato y estrénalo con un día nuevo

Ya viste que un árbol que hace **todas** las preguntas que quiere memoriza el azar. Ahora crea uno **sensato**, compáralo con el memorión y úsalo para decidir. Usa los datos de los 200 días (`X_ent`, `y_ent`, `X_exa`, `y_exa`):

1. Crea `arbol_sensato`: un `DecisionTreeClassifier` con `max_depth=3` y `min_samples_leaf=10` (cada hoja debe reunir al menos 10 días, así no memoriza) y entrénalo con `X_ent`, `y_ent`.
2. Crea `arbol_memorion` **sin límites** (solo `random_state=42`) y entrénalo igual.
3. Imprime, para cada uno, cuántas **hojas** tiene (`get_n_leaves()`) y cuánto **acierta en el examen** (`score(X_exa, y_exa)`).
4. Llega un día nuevo: **85 %** de nubes y **90 %** de humedad. Créalo como tabla (`dia_nuevo`) con las mismas columnas que `X` y pregunta a tu árbol sensato si va a llover.
5. Comprueba que el sensato tiene menos hojas que el memorión y que para el día nuevo predice que **sí llueve** (1).

In [ ]:
# Escribe tu código aquí

# 1) Tu arbol sensato
# arbol_sensato = DecisionTreeClassifier(max_depth=..., min_samples_leaf=..., random_state=42).fit(X_ent, y_ent)

# 2) El memorion (sin limites)
# arbol_memorion = DecisionTreeClassifier(random_state=42).fit(X_ent, y_ent)

# 3) Comparar hojas y aciertos en el examen
# for nombre, a in [("sensato", arbol_sensato), ("memorión", arbol_memorion)]:
#     print(nombre, a.get_n_leaves(), a.score(X_exa, y_exa))

# 4) Un dia nuevo (las columnas deben llamarse igual que en X)
# dia_nuevo = pd.DataFrame({"nubes (%)": [85], "humedad (%)": [90]})
# prediccion = arbol_sensato.predict(dia_nuevo)[0]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Tu arbol sensato
arbol_sensato = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42).fit(X_ent, y_ent)

# 2) El memorion (sin limites)
arbol_memorion = DecisionTreeClassifier(random_state=42).fit(X_ent, y_ent)

# 3) Comparar hojas y aciertos en el examen
for nombre, a in [("sensato ", arbol_sensato), ("memorión", arbol_memorion)]:
    print(f"Árbol {nombre}: {a.get_n_leaves():2d} hojas | acierta {a.score(X_exa, y_exa):.0%} en el examen")

# 4) Un dia nuevo
dia_nuevo = pd.DataFrame({"nubes (%)": [85], "humedad (%)": [90]})
prediccion = arbol_sensato.predict(dia_nuevo)[0]
print("\nDía nuevo (85 % nubes, 90 % humedad):", "¡lleva paraguas, va a llover!" if prediccion == 1 else "no hace falta paraguas")

# 5) Comprobaciones
assert arbol_sensato.get_n_leaves() < arbol_memorion.get_n_leaves()
assert prediccion == 1
print("\nEl árbol sensato tiene muchas menos hojas, no memoriza el azar y acierta con el día nuevo.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Un árbol de decisión es un juego de preguntas de sí o no que el computador aprende solo, eligiendo en cada paso la pregunta que más «ordena» los casos; es fácil de leer, pero si tiene demasiadas preguntas memoriza y si cambian pocos datos puede cambiar por completo.**

### Lo que aprendiste hoy:
- ✅ Un árbol es una **cadena de preguntas** con respuestas finales (hojas) que se pueden **leer**.
- ✅ Elige la pregunta que deja los grupos **más ordenados** y repite dentro de cada grupo.
- ✅ **Demasiadas preguntas = memorizar**: acierta lo que ya vio, falla en lo nuevo.
- ✅ Un solo árbol es **inestable**: por eso luego se usan **muchos árboles juntos**.

> 🎓 **Siguiente paso:** [Random Forests (Dummies)](01_Random_Forests_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../00_Arboles_de_Decision.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué un árbol que acierta el 100 % de los datos que ya vio puede ser **peor** que uno más sencillo?
2. Piensa en una decisión de tu vida diaria o de tu trabajo que se pueda tomar con 3 preguntas de sí/no (¿pido un préstamo?, ¿salgo hoy en moto?). Escribe tu «árbol»: ¿cuál sería la **primera** pregunta y por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>